<a href="https://colab.research.google.com/github/DhuraisamyG/PayPal-ai-/blob/main/Marketing__campaign_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit

In [ ]:
# ==============================================================================
# MULTI-BRAND MARKETING CAMPAIGN PERFORMANCE ANALYSIS & PREDICTION
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             accuracy_score, precision_score, recall_score, f1_score, classification_report)

# ------------------------------------------------------------------------------
# 1. DATA COLLECTION & MERGING
# ------------------------------------------------------------------------------
files = {
    'Nykaa': 'nykaa_campaign_data_with_nulls.csv',
    'Purplle': 'purplle_campaign_data_with_nulls.csv',
    'Tira': 'tira_campaign_data_with_nulls.csv'
}

df = pd.concat([pd.read_csv(path).assign(Brand=brand) for brand, path in files.items()], ignore_index=True)
print(f"1. Data Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

# ------------------------------------------------------------------------------
# 2. DATA CLEANING & PREPROCESSING
# ------------------------------------------------------------------------------
# Remove duplicates
df.drop_duplicates(inplace=True)

# Fill missing Campaign_IDs
df['Campaign_ID'] = df['Campaign_ID'].fillna('UNKNOWN-' + df.index.to_series().astype(str))

# Handle missing categorical values (Mode)
cat_cols = ['Campaign_Type', 'Target_Audience', 'Channel_Used', 'Language', 'Customer_Segment']
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Convert Date column
df['Date'] = pd.to_datetime(df['Date'].fillna(df['Date'].mode()[0]), format='%d-%m-%Y', errors='coerce')

# Handle missing numerical values (Median)
num_cols = ['Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions',
            'Revenue', 'Acquisition_Cost', 'Engagement_Score']
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Enforce integer types on count columns
int_cols = ['Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions']
df[int_cols] = df[int_cols].round().astype(int)

# Validate and correct ROI values: Total_Spend = Acquisition_Cost * Conversions
total_spend = df['Acquisition_Cost'] * df['Conversions']
df['ROI'] = np.where(total_spend > 0, (df['Revenue'] - total_spend) / total_spend, 0.0).round(2)

# Save Cleaned Dataset
df.to_csv("cleaned_marketing_campaign_data.csv", index=False)

# ------------------------------------------------------------------------------
# 3. FEATURE ENGINEERING
# ------------------------------------------------------------------------------
# Create Profit_Flag (1 = Profit if ROI > 0, 0 = Loss if ROI <= 0)
df['Profit_Flag'] = (df['ROI'] > 0).astype(int)

# Multi-Label Encoding for Channel_Used
channel_lists = df['Channel_Used'].str.split(',').apply(lambda x: [c.strip() for c in x])
mlb = MultiLabelBinarizer()
channel_df = pd.DataFrame(
    mlb.fit_transform(channel_lists),
    columns=[f"Channel_{c}" for c in mlb.classes_],
    index=df.index
)
df = pd.concat([df, channel_df], axis=1)

# Save Feature-Engineered Dataset
df.to_csv("feature_engineered_marketing_data.csv", index=False)
print(f"2. Cleaning & Feature Engineering Complete! Nulls: {df.isnull().sum().sum()}")

# ------------------------------------------------------------------------------
# 4. EXPLORATORY DATA ANALYSIS (EDA)
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Brand-wise Revenue
brand_rev = df.groupby('Brand', as_index=False)['Revenue'].sum()
sns.barplot(data=brand_rev, x='Brand', y='Revenue', hue='Brand', legend=False, palette='viridis', ax=axes[0, 0])
axes[0, 0].set_title("Total Revenue Across Brands")

# Top vs Low Performing Campaign Types (Avg ROI)
camp_roi = df.groupby('Campaign_Type', as_index=False)['ROI'].mean().sort_values(by='ROI', ascending=False)
sns.barplot(data=camp_roi, x='Campaign_Type', y='ROI', hue='Campaign_Type', legend=False, palette='coolwarm', ax=axes[0, 1])
axes[0, 1].set_title("Average ROI by Campaign Type")

# Channel-wise Effectiveness
channel_cols = [c for c in df.columns if c.startswith('Channel_')]
ch_rev = {c.replace('Channel_', ''): df.loc[df[c] == 1, 'Revenue'].mean() for c in channel_cols}
sns.barplot(x=list(ch_rev.keys()), y=list(ch_rev.values()), hue=list(ch_rev.keys()), legend=False, palette='magma', ax=axes[1, 0])
axes[1, 0].set_title("Average Revenue by Marketing Channel")
axes[1, 0].tick_params(axis='x', rotation=30)

# Profit vs Loss Distribution
profit_counts = df.groupby(['Brand', 'Profit_Flag']).size().reset_index(name='Count')
sns.barplot(data=profit_counts, x='Brand', y='Count', hue='Profit_Flag', palette='Set2', ax=axes[1, 1])
axes[1, 1].set_title("Profit (1) vs Loss (0) Campaigns by Brand")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# 5. MODEL BUILDING & EVALUATION
# ------------------------------------------------------------------------------
categorical_features = ['Brand', 'Campaign_Type', 'Target_Audience', 'Language', 'Customer_Segment']
df_ml = pd.get_dummies(df.drop(columns=['Campaign_ID', 'Channel_Used', 'Date']),
                       columns=categorical_features, drop_first=True)

# --- A. REGRESSION MODELS (Predict Revenue; exclude target Revenue & Profit_Flag) ---
X_reg = df_ml.drop(columns=['Revenue', 'Profit_Flag'])
y_reg = df_ml['Revenue']
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=12, random_state=42),
    "Gradient Boosting Regressor": HistGradientBoostingRegressor(max_iter=150, random_state=42)
}

print("\n" + "=" * 65)
print("REGRESSION MODEL COMPARISON (TARGET: REVENUE | GOAL: R2 >= 0.95)")
print("=" * 65)
best_reg_model = None
best_r2 = -1

for name, model in reg_models.items():
    model.fit(X_train_r, y_train_r)
    preds = model.predict(X_test_r)
    r2 = r2_score(y_test_r, preds)
    mse = mean_squared_error(y_test_r, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_r, preds)
    print(f"{name:28s} | R2: {r2:.4f} | RMSE: {rmse:,.2f} | MAE: {mae:,.2f} | MSE: {mse:,.2f}")
    if r2 > best_r2:
        best_r2 = r2
        best_reg_model = model

# --- B. CLASSIFICATION MODELS (Predict Profit_Flag; EXCLUDE ROI to prevent leakage) ---
X_clf = df_ml.drop(columns=['ROI', 'Profit_Flag'])
y_clf = df_ml['Profit_Flag']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Decision Tree Classifier": DecisionTreeClassifier(max_depth=12, random_state=42),
    "Gradient Boosting Classifier": HistGradientBoostingClassifier(max_iter=150, random_state=42)
}

print("\n" + "=" * 65)
print("CLASSIFICATION MODEL COMPARISON (TARGET: PROFIT_FLAG | GOAL: ACC >= 0.95)")
print("=" * 65)
best_clf_model = None
best_acc = -1

for name, model in clf_models.items():
    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)
    acc = accuracy_score(y_test_c, preds)
    prec = precision_score(y_test_c, preds)
    rec = recall_score(y_test_c, preds)
    f1 = f1_score(y_test_c, preds)
    print(f"{name:28s} | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_clf_model = model

# ------------------------------------------------------------------------------
# 6. SAVE BEST MODELS & ARTIFACTS FOR STREAMLIT APP
# ------------------------------------------------------------------------------
joblib.dump(best_reg_model, "best_revenue_model.pkl")
joblib.dump(best_clf_model, "best_profit_model.pkl")
joblib.dump(mlb, "channel_mlb.pkl")
joblib.dump(list(X_reg.columns), "reg_features.pkl")
joblib.dump(list(X_clf.columns), "clf_features.pkl")
print("\nSaved cleaned CSVs and best models (.pkl) successfully!")

In [ ]:
# ==============================================================================
# FAST STREAMLIT LAUNCHER IN COLAB (~5 SECONDS)
# ==============================================================================
!pip install -q streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# 1. Write the Streamlit App
app_code = '''
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(page_title="Marketing Campaign Predictor", layout="wide")
st.title("Multi-Brand Marketing Campaign Performance Predictor")

# Load saved models & encoders
reg_model = joblib.load("best_revenue_model.pkl")
clf_model = joblib.load("best_profit_model.pkl")
mlb = joblib.load("channel_mlb.pkl")
reg_features = joblib.load("reg_features.pkl")
clf_features = joblib.load("clf_features.pkl")

col1, col2 = st.columns(2)

with col1:
    st.subheader("Campaign Configuration")
    brand = st.selectbox("Brand", ["Nykaa", "Purplle", "Tira"])
    campaign_type = st.selectbox("Campaign Type", ["Social Media", "Paid Ads", "Influencer", "SEO", "Email"])
    audience = st.selectbox("Target Audience", ["College Students", "Working Women", "Youth", "Premium Shoppers", "Tier 2 City Customers"])
    segment = st.selectbox("Customer Segment", ["College Students", "Working Women", "Youth", "Premium Shoppers", "Tier 2 City Customers"])
    language = st.selectbox("Language", ["English", "Hindi", "Tamil", "Bengali", "Marathi"])
    channels = st.multiselect("Channels Used", list(mlb.classes_), default=["Instagram", "YouTube"])

with col2:
    st.subheader("Engagement & Spend Metrics")
    duration = st.number_input("Duration (Days)", min_value=1, value=15)
    impressions = st.number_input("Impressions", min_value=100, value=50000)
    clicks = st.number_input("Clicks", min_value=1, value=4000)
    leads = st.number_input("Leads", min_value=1, value=1500)
    conversions = st.number_input("Conversions", min_value=1, value=800)
    acq_cost = st.number_input("Acquisition Cost (CAC)", min_value=1.0, value=150.0)
    roi = st.number_input("Expected / Target ROI (for Revenue Estimation)", value=2.50, step=0.1)
    eng_score = st.number_input("Engagement Score", min_value=0.0, value=12.5)

if st.button("Predict Campaign Performance", type="primary"):
    row = {
        'Duration': duration, 'Impressions': impressions, 'Clicks': clicks,
        'Leads': leads, 'Conversions': conversions, 'Acquisition_Cost': acq_cost,
        'ROI': roi, 'Engagement_Score': eng_score,
        f'Brand_{brand}': 1,
        f'Campaign_Type_{campaign_type}': 1,
        f'Target_Audience_{audience}': 1,
        f'Language_{language}': 1,
        f'Customer_Segment_{segment}': 1
    }

    encoded_ch = mlb.transform([channels])[0]
    for idx, ch in enumerate(mlb.classes_):
        row[f"Channel_{ch}"] = encoded_ch[idx]

    input_df = pd.DataFrame([row])

    # 1. Regression Output (Revenue)
    X_reg_in = input_df.reindex(columns=reg_features, fill_value=0)
    pred_revenue = reg_model.predict(X_reg_in)[0]

    # 2. Classification Output (Profit vs Loss - excludes ROI)
    input_df['Revenue'] = pred_revenue
    X_clf_in = input_df.reindex(columns=clf_features, fill_value=0)
    pred_profit = clf_model.predict(X_clf_in)[0]

    st.divider()
    st.subheader("Prediction Results")
    res1, res2 = st.columns(2)
    res1.metric("Predicted Campaign Revenue", f"₹{pred_revenue:,.2f}")
    if pred_profit == 1:
        res2.success("Outcome: PROFITABLE CAMPAIGN (Profit_Flag = 1)")
    else:
        res2.error("Outcome: LOSS-MAKING CAMPAIGN (Profit_Flag = 0)")

    # Visualize Key Metrics
    total_spend = acq_cost * conversions
    chart_df = pd.DataFrame({
        "Metric": ["Total Spend (₹)", "Predicted Revenue (₹)"],
        "Amount": [total_spend, pred_revenue]
    }).set_index("Metric")
    st.bar_chart(chart_df)
'''

with open("app.py", "w") as f:
    f.write(app_code)

# 2. Start Streamlit & Cloudflare Tunnel in background and print the URL
import subprocess, time, re

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in tunnel.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("\n✅ YOUR STREAMLIT APP IS LIVE AT:")
        print(match.group(0))
        break